# Uplift Modeling em Marketing — Notebook 4: Causal Forest e Uplift Trees (S5)

> Continuação de `03_Meta_Learners_PT.ipynb`. Este notebook não compartilha o
> kernel do anterior — recarrega abaixo os splits de treino/validação. O
> Notebook 03 e a Seção 4 permanecem congelados: nenhuma célula ali foi
> alterada nesta rodada. O teste selado continua oculto: este notebook nunca
> o toca.
>
> **Objetivo de S5:** comparar estimadores causais e árvores especializadas em uplift (Causal
> Forest, Uplift Tree, Uplift Random Forest) contra as referências
> sobreviventes de S4 (X-learner + árvore rasa, S-learner + LightGBM vanilla)
> e o baseline de propensão, usando exatamente o protocolo de desenvolvimento
> que S4 estabeleceu — repeated stratified holdout dentro de `train_df`,
> avaliação complementar única em `val_df`. **S5 não escolhe a configuração
> final** — isso fica para o término desta seção, com revisão explícita antes
> de congelar e abrir o teste selado em S6.


## Índice

- [Setup — retomando de S1-S4](#setup)
- [Seção 5 — Causal Forest e Uplift Trees](#s5)
    - [5.1 Protocolo e hipóteses pré-registradas](#s5-1)
    - [5.2 Implementação e sanity checks](#s5-2)
    - [5.3 Repeated stratified holdout — comparação principal](#s5-3)
    - [5.4 Diferenças pareadas contra baseline e referências de S4](#s5-4)
    - [5.5 Avaliação complementar no holdout fixo](#s5-5)
    - [5.6 Correlação entre rankings](#s5-6)
    - [5.7 Síntese da S5](#s5-7)
    - [5.8 Decisão pré-S6 — tomada após revisão de S5 e antes da abertura do teste](#s5-8)

---

In [1]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.exceptions import ConvergenceWarning

# Bootstrap de path: permite `from src...` a partir de notebooks/
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import SEED
from src.i18n import make_lang
from src.viz import apply_plot_style

np.random.seed(SEED)
pd.set_option('display.max_columns', 50)
pd.set_option('display.precision', 4)
warnings.filterwarnings('ignore', category=FutureWarning)
# CausalForestDML (model_y='auto') inclui um WeightedLassoCVWrapper (solver
# SAG) entre os candidatos de primeiro estágio — não converge dentro do
# max_iter padrão em bases deste tamanho; não afeta o resultado (é só um dos
# modelos candidatos do nuisance model), mas polui a saída.
warnings.filterwarnings('ignore', category=ConvergenceWarning)

lang = make_lang('pt')
apply_plot_style()


In [2]:
# Stack causal de S5 — verificado contra a API real instalada nesta rodada
# (não suposto de memória/documentação de outra versão).
import importlib.metadata as importlib_metadata

from causalml.inference.tree import UpliftRandomForestClassifier, UpliftTreeClassifier
from econml.dml import CausalForestDML
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeRegressor

print('Versões instaladas realmente usadas nesta rodada:')
print(f"  econml:   {importlib_metadata.version('econml')}")
print(f"  causalml: {importlib_metadata.version('causalml')}")


Failed to import duecredit due to No module named 'duecredit'


Versões instaladas realmente usadas nesta rodada:
  econml:   0.16.0
  causalml: 0.15.5


<a id="setup"></a>

## Setup — retomando de S1-S4

No fluxo normal, `get_train_val` carrega os manifests de treino/validação já
persistidos por `02_Baseline_Propensity_PT` (`train_index.parquet`,
`validation_index.parquet`, `dataset_manifest.json`) e valida o fingerprint
SHA-256 e `n_rows` do dataset atual contra o manifest — sem reescrever o
índice do teste selado, que só o notebook 2 grava. O tratamento, o outcome
primário e o conjunto original de covariáveis são exatamente os mesmos de
S3/S4: `treatment` pooled (`any email` vs. `No E-Mail`), outcome `visit`,
`FEATURE_COLS` de `src/config.py`. Não voltamos a separar Men's/Women's
E-Mail nesta seção — seria outro estimand.


In [3]:
from src.config import FEATURE_COLS, POOLED_TREATMENT_COL, PRIMARY_OUTCOME
from src.data import add_pooled_treatment, load_hillstrom
from src.splits import get_train_val

df = load_hillstrom()
df_pooled = add_pooled_treatment(df)
train_df, val_df = get_train_val(df_pooled, persist_test=False)

labels = lang({'header': 'Tamanho das partições (retomado de S1-S4)'})
print(f"{labels['header']}:")
print(f"  Treino:     {len(train_df):>6} linhas | tratados: {int(train_df[POOLED_TREATMENT_COL].sum()):>6}")
print(f"  Validação:  {len(val_df):>6} linhas | tratados: {int(val_df[POOLED_TREATMENT_COL].sum()):>6}")
print(f"  Features:   {FEATURE_COLS}")


Tamanho das partições (retomado de S1-S4):
  Treino:      38400 linhas | tratados:  25617
  Validação:   12800 linhas | tratados:   8538
  Features:   ['recency', 'history', 'mens', 'womens', 'newbie', 'zip_code', 'channel', 'history_segment']


<a id='s5'></a>
<a id="s5"></a>

## Seção 5 — Causal Forest e Uplift Trees

S4 não produziu um vencedor único: X-learner + árvore rasa e S-learner +
LightGBM vanilla formam um top tier sem separação estável sob repeated
holdout, e nenhum dos dois bate o baseline de propensão de forma consistente
(4.9.3). Essa seção testa se algum estimador causal (`CausalForestDML`, que
ortogonaliza via nuisance models — inclusive um `model_y` de outcome) ou
árvore especializada em uplift (`UpliftTree`/`UpliftRF`, que constroem
splits orientados diretamente a diferenças de resposta entre tratamento e
controle) consegue produzir um ranking incremental que sobreviva à troca de amostra.
S5 não é uma corrida para superar o Qini AUC 0.0627 do holdout fixo de S4.6;
é um teste sobre estabilidade sob reamostragem, no mesmo espírito de 4.9.


<a id="s5-1"></a>

### 5.1 Protocolo e hipóteses pré-registradas

**Referências congeladas de S4, obrigatórias na comparação:** X-learner +
`DecisionTreeRegressor(max_depth=4)` (top tier, holdout fixo 0.0627, média
0.0250 no repeated holdout de 4.9.2); S-learner + LightGBM vanilla (top tier,
média 0.0232); baseline de propensão de resposta — `fit_propensity_baseline`,
que estima P(Y=1|X,T=1) (propensão de **resposta**, o modelo que marketing
já treinaria sem noção de efeito incremental), não P(T=1|X) (propensão de
**assignment**, irrelevante aqui porque o desenho é um RCT). O nome histórico
"Baseline (propensão)" é preservado nas tabelas por continuidade com S4.

**Protocolo principal — idêntico a 4.9:** repeated stratified holdout dentro
de `train_df` (nunca `val_df` nem o teste selado), 15 repetições, split
75% fit / 25% evaluation, estratificado por (tratamento pooled × `visit`),
seeds `1000+rep`, os mesmos `fit_idx`/`eval_idx` reusados entre todos os
candidatos de uma mesma repetição — permite diferenças pareadas por split.
Antes de interpretar qualquer resultado novo, verificamos se os três
benchmarks congelados reproduzem os números de 4.9.2/4.9.3 (X+Tree≈0.0250,
S+LightGBM≈0.0232, baseline≈0.0209) — como splits, seeds, dados e
implementações são os mesmos, essa reprodução é esperada de forma exata, não
aproximada; uma discrepância material sinalizaria uma regressão de
implementação, não um resultado novo.

**Hipóteses registradas antes de rodar** — nenhuma com expectativa de
superioridade automática:

**Causal Forest.** Por estimar heterogeneidade diretamente e usar
ortogonalização/nuisance models, pode produzir ranking mais estável que
alguns meta-learners, mas não há hipótese de que necessariamente supere as
referências de S4.

**Uplift Tree.** Ao construir splits voltados diretamente à separação de
efeito (não à predição de outcome), pode encontrar segmentos úteis que
modelos de outcome tradicionais não priorizam; em contrapartida, uma árvore
única pode apresentar alta variância — o mesmo padrão de instabilidade já
visto no T-learner de 4.9.2.

**Uplift Random Forest.** A agregação de árvores pode estabilizar o ranking
do Uplift Tree, mas S4 já mostrou (4.6→4.8) que bagging não garante melhoria
de Qini — é uma hipótese a testar, não uma expectativa de superioridade.

**Critério de convicção, registrado antes de ver qualquer resultado:** o
resultado mais convincente é aquele que mantém desempenho relativo sob
repeated holdout; um Qini elevado apenas no `val_df` fixo não é suficiente —
S4 já mostrou exatamente esse padrão (T+Tree: 0.0615 no holdout fixo, 0.0039
no repeated holdout).


<a id="s5-2"></a>

### 5.2 Implementação e sanity checks

**API real inspecionada nesta rodada** (não suposta de memória): `econml`
0.16.0, `causalml` 0.15.5 — versões impressas na célula de import acima.

**`CausalForestDML`.** Dois parâmetros não-default, ambos exigidos pelo
desenho do estudo: `discrete_treatment=True` (tratamento pooled é binário
{0,1}, não contínuo) e `model_t=DummyClassifier(strategy='prior')`. Este
último existe porque este é um RCT — o mecanismo de atribuição de tratamento
não depende de X. O default `model_t='auto'` expande, via inspeção do
código-fonte de `econml.dml.dml._make_first_stage_selector`/`get_selector`,
para um `GridSearchCV` sobre `RandomForestClassifier`/`LogisticRegressionCV`:
um nuisance model flexível que aprenderia variação espúria de P(T=1) em
função de X sob um mecanismo que é, por desenho, X-invariante. Um estimador
sklearn-compatível passado diretamente (não `'auto'`/lista/string) cai no
ramo `FixedModelSelector` do seletor — usado como está, sem grid search por
cima; não foi preciso nenhum hack para forçar a propensão conhecida (2/3) na
API. `model_y` fica no default `'auto'`, consistente com o resto do projeto,
que sempre regride a probabilidade do outcome binário (nunca classifica).

**`UpliftTreeClassifier`/`UpliftRandomForestClassifier`.** Ambos exigem
`treatment` como rótulos de grupo (string), não 0/1 — `control_name`
identifica explicitamente qual rótulo é o controle. Os dois `predict()` têm
semânticas **diferentes**, confirmadas por inspeção do `.pyx` fonte
instalado, não presumidas: `UpliftTreeClassifier.predict(X)` retorna
probabilidades P(Y=1) por grupo (uma coluna por `classes_`, controle
incluso) — não é uplift diretamente, precisa da subtração
`P(Y=1|tratado) − P(Y=1|controle)`, localizando as colunas via
`classes_.index(...)`, nunca por posição fixa.
`UpliftRandomForestClassifier.predict(X, full_output=True)` já calcula
`delta_{grupo}` explicitamente — usamos essa coluna nomeada, não a saída
posicional de `full_output=False` (que também é o uplift, mas sem rótulo).

**Hiperparâmetros não-default, documentados antes de qualquer resultado**
(nenhum grid search, nenhum ajuste depois de ver Qini): `discrete_treatment`
e `model_t` do `CausalForestDML` (acima); `control_name` e `random_state` do
`UpliftTreeClassifier`/`UpliftRandomForestClassifier` (exigência técnica da
API + reprodutibilidade). Todo o resto fica no default da biblioteca —
`n_estimators=100`/`cv=2`/`honest=True` (Causal Forest), `max_depth=3` (Tree),
`n_estimators=10`/`max_depth=5` (RF), `evaluationFunction='KL'` (ambos).

**Preprocessing sem leakage.** Os três novos candidatos reusam
`encode_meta_learner_features`/`build_meta_learner_encoder` — a mesma
infraestrutura de S4, já com a garantia estrutural de que o encoder é
ajustado somente no `fit_df` de cada repetição, nunca antes do split (ver
`repeated_stratified_holdout` em `src/evaluation.py`).

**Smoke tests** (dados sintéticos com tratamento randomizado e efeito
heterogêneo conhecido, e uma amostra real pequena em memória — nunca o teste
selado) confirmaram, para os três wrappers: comprimento correto, valores
finitos, score não-constante, sinal correto (correlação positiva com o
efeito verdadeiro) e reprodutibilidade bit-a-bit sob o mesmo `random_state`
— ver `tests/test_suite.py`.

**Custo computacional.** Em smoke tests isolados, com dados no tamanho real
de um `fit_df` de repeated holdout (~28.800 linhas × 18 colunas one-hot), um
fit de `CausalForestDML` com os hiperparâmetros default chegou a ~207s —
bem mais caro que os demais candidatos (segundos), o que motivou cautela
quanto ao tempo total esperado da célula de 5.3 abaixo. Na execução
completa final de 5.3, porém, os 15 repeated holdouts contendo os seis
candidatos totalizaram aproximadamente 781,5s (~13 min) — bem abaixo do que
uma extrapolação ingênua a partir do smoke test isolado sugeriria, mostrando
variação relevante de custo entre execuções/condições do ambiente.
`UpliftTreeClassifier`/`UpliftRandomForestClassifier` são baratos (segundos).


In [4]:
# Configuração explícita de cada família — registrada antes de qualquer
# resultado (nenhum grid search, nenhum ajuste posterior). Impressos a
# partir dos próprios argumentos passados ao construtor — nem
# `CausalForestDML` nem `UpliftTreeClassifier`/`UpliftRandomForestClassifier`
# expõem `.get_params()` (verificado: `AttributeError`), diferente da
# convenção usual de estimadores scikit-learn.
labels = lang({'header': 'Parâmetros não-default de cada família (o resto é default da biblioteca)'})
print(f"{labels['header']}:\n")
print('CausalForestDML: discrete_treatment=True, model_t=DummyClassifier(strategy=\'prior\'), random_state=%r' % SEED)
print('UpliftTreeClassifier: control_name=\'control\', random_state=%r' % SEED)
print('UpliftRandomForestClassifier: control_name=\'control\', random_state=%r' % SEED)


Parâmetros não-default de cada família (o resto é default da biblioteca):

CausalForestDML: discrete_treatment=True, model_t=DummyClassifier(strategy='prior'), random_state=42
UpliftTreeClassifier: control_name='control', random_state=42
UpliftRandomForestClassifier: control_name='control', random_state=42


<a id="s5-3"></a>

### 5.3 Repeated stratified holdout — comparação principal

Seis candidatos, os mesmos 15 splits/seeds reusados entre todos (dentro de
`train_df`, nunca `val_df` nem o teste selado): as duas referências
congeladas de S4, o baseline de propensão, e os três estimadores
especializados de efeito/uplift desta seção.


In [5]:
from src.evaluation import paired_deltas, repeated_holdout_summary, repeated_stratified_holdout

tree_factory = lambda: DecisionTreeRegressor(max_depth=4, random_state=SEED)

s5_candidates = {
    'X+Tree(depth=4)': ('meta', 'X', tree_factory, False),
    'S+LightGBM(vanilla)': ('meta', 'S', None, False),
    'Baseline (propensão)': ('propensity', None, None, False),
    'CausalForest': ('causal_forest', None, None, False),
    'UpliftTree': ('uplift_tree', None, None, False),
    'UpliftRF': ('uplift_rf', None, None, False),
}

t0 = time.time()
s5_results = repeated_stratified_holdout(train_df, POOLED_TREATMENT_COL, PRIMARY_OUTCOME, s5_candidates, n_reps=15)
elapsed = time.time() - t0
s5_summary = repeated_holdout_summary(s5_results)

labels = lang({'header': 'Repeated stratified holdout — 15 splits (só em train_df)'})
print(f"{labels['header']} — tempo total: {elapsed:.1f}s:")
print(s5_summary.round(4))


Repeated stratified holdout — 15 splits (só em train_df) — tempo total: 781.5s:
                        mean  median     std     min     max  win_rate
candidate                                                             
UpliftTree            0.0254  0.0201  0.0141  0.0061  0.0537    0.4000
X+Tree(depth=4)       0.0250  0.0259  0.0165 -0.0044  0.0437    0.1333
S+LightGBM(vanilla)   0.0232  0.0228  0.0115 -0.0016  0.0433    0.2000
Baseline (propensão)  0.0209  0.0172  0.0149 -0.0004  0.0459    0.2000
UpliftRF              0.0201  0.0204  0.0096  0.0045  0.0348    0.0000
CausalForest          0.0131  0.0121  0.0163 -0.0189  0.0429    0.0667


**Insights:**

Os três benchmarks congelados reproduziram exatamente os números de 4.9.2/4.9.3 — X+Tree(depth=4) média 0.0250, S+LightGBM(vanilla) média 0.0232, baseline média 0.0209 — como esperado, já que splits, seeds, dados e implementações são os mesmos; nenhuma regressão de implementação detectada.

**UpliftTree tem a maior média entre os seis candidatos (0.0254), à frente até de X+Tree (0.0250), e a maior taxa de vitória (40,0% dos splits).** Mas sua mediana (0.0201) fica abaixo da de X+Tree (0.0259) — uma distribuição mais assimétrica à direita, com um teto mais alto (máximo 0.0537, o maior dos seis) puxando a média para cima, enquanto o split "típico" fica um pouco atrás de X+Tree. UpliftRF fica perto do baseline em média (0.0201 vs. 0.0209) e tem o menor desvio-padrão de todos os seis (0.0096) — o candidato numericamente mais estável — mas nunca é o melhor candidato em nenhum dos 15 splits (taxa de vitória 0%), um perfil de "meio-termo consistente" em vez de líder ocasional.

CausalForest tem a menor média (0.0131) e uma dispersão ampla (desvio-padrão 0.0163), incluindo pelo menos um split com Qini AUC negativo (-0.0189) — desempenho abaixo de um ranking aleatório naquela reamostragem específica. Nenhum dos três estimadores especializados de efeito/uplift estabeleceu uma vantagem estável e unidirecional sobre as referências congeladas de S4 nesta comparação.

<a id="s5-4"></a>

### 5.4 Diferenças pareadas contra baseline e referências de S4

Para cada um dos três estimadores especializados de efeito/uplift: Δ contra o baseline de propensão,
Δ contra X+Tree(depth=4), Δ contra S+LightGBM(vanilla) — mesmos splits,
diferença calculada split a split (não médias agregadas isoladas).


In [6]:
new_candidates = ['CausalForest', 'UpliftTree', 'UpliftRF']

deltas_vs_baseline = paired_deltas(s5_results, baseline_candidate='Baseline (propensão)').loc[new_candidates]
deltas_vs_xtree = paired_deltas(s5_results, baseline_candidate='X+Tree(depth=4)').loc[new_candidates]
deltas_vs_slgbm = paired_deltas(s5_results, baseline_candidate='S+LightGBM(vanilla)').loc[new_candidates]

labels = lang({
    'h1': 'Δ = Qini(candidato) − Qini(baseline de propensão), por split',
    'h2': 'Δ = Qini(candidato) − Qini(X+Tree(depth=4)), por split',
    'h3': 'Δ = Qini(candidato) − Qini(S+LightGBM(vanilla)), por split',
})
print(f"{labels['h1']}:")
print(deltas_vs_baseline.round(4))
print(f"\n{labels['h2']}:")
print(deltas_vs_xtree.round(4))
print(f"\n{labels['h3']}:")
print(deltas_vs_slgbm.round(4))


Δ = Qini(candidato) − Qini(baseline de propensão), por split:
              delta_mean  delta_median  delta_std  prop_delta_positive
CausalForest     -0.0077       -0.0054     0.0194               0.2667
UpliftTree        0.0045        0.0077     0.0189               0.6667
UpliftRF         -0.0007        0.0038     0.0161               0.6000

Δ = Qini(candidato) − Qini(X+Tree(depth=4)), por split:
              delta_mean  delta_median  delta_std  prop_delta_positive
CausalForest     -0.0119       -0.0091     0.0240               0.4000
UpliftTree        0.0004        0.0001     0.0207               0.5333
UpliftRF         -0.0049       -0.0044     0.0174               0.2667

Δ = Qini(candidato) − Qini(S+LightGBM(vanilla)), por split:
              delta_mean  delta_median  delta_std  prop_delta_positive
CausalForest     -0.0101       -0.0142     0.0118               0.2000
UpliftTree        0.0022       -0.0020     0.0190               0.4667
UpliftRF         -0.0031       -0.0019 

**Insights:**

**Contra o baseline de propensão, UpliftTree mostra o sinal mais consistente dos três** — vence em 10 de 15 splits (66,67%), com delta mediano positivo (+0,0077); apenas uma repetição de diferença dos 9 de 15 (60%) observados para S+LightGBM contra esse mesmo baseline em 4.9.3. UpliftRF também supera o baseline na maioria dos splits (60,0%), com delta mediano positivo (+0,0038) apesar de um delta médio ligeiramente negativo (-0,0007) — o mesmo padrão de assimetria já visto em 5.3, uns poucos splits ruins puxando a média para baixo enquanto o split típico favorece UpliftRF. CausalForest perde para o baseline na maioria dos splits (vence em só 26,67%), com deltas médio e mediano negativos — o mais fraco dos três nesta comparação.

Contra X+Tree(depth=4), UpliftTree apresenta praticamente empate descritivo no protocolo repetido: Δ médio +0,0004, Δ mediano +0,0001 e 8/15 deltas positivos (53,33%). CausalForest e UpliftRF apresentam proporções menores de deltas positivos, 6/15 (40,0%) e 4/15 (26,67%), respectivamente — o mesmo padrão de "sem separação estável" já registrado entre X+Tree e S+LightGBM em 4.9.2 (53%). Contra S+LightGBM(vanilla), UpliftTree novamente fica perto do empate (46,67%), enquanto UpliftRF e CausalForest pendem para o lado negativo (33,33% e 20,0%).

Resumindo o padrão: UpliftTree é o único dos três estimadores especializados de efeito/uplift com um sinal direcionalmente positivo e razoavelmente consistente — especificamente contra o baseline —, sem separação estável, no protocolo repetido, das duas referências de top tier de S4. UpliftRF mostra um padrão semelhante, porém mais fraco. CausalForest não mostra sinal positivo em nenhuma das três comparações pareadas.

<a id="s5-5"></a>

### 5.5 Avaliação complementar no holdout fixo

Uma única avaliação, `train_df` completo → `val_df`, com as configurações já
congeladas dos três estimadores especializados de efeito/uplift (nenhum
ajuste feito depois de ver este resultado). Complementar ao repeated holdout, não substituto — S4 já
mostrou por que um único holdout pode enganar (T+Tree: 0.0615 no holdout
fixo, 0.0039 no repeated holdout). Um resultado espetacular aqui e medíocre
no repeated holdout acima seria lido como instabilidade, não como vitória.


In [7]:
from src.evaluation import evaluate_multiple_rankings
from src.learners import (
    build_meta_learner_encoder, encode_meta_learner_features, fit_causal_forest,
    fit_propensity_baseline, fit_single_meta_learner, fit_uplift_random_forest, fit_uplift_tree,
    predict_causal_forest_uplift, predict_propensity_score, predict_single_meta_learner,
    predict_uplift_random_forest_uplift, predict_uplift_tree_uplift,
)

encoder_s5 = build_meta_learner_encoder(train_df)
X_train_s5 = encode_meta_learner_features(train_df, encoder_s5)
X_val_s5 = encode_meta_learner_features(val_df, encoder_s5)
treatment_train_s5 = train_df[POOLED_TREATMENT_COL].to_numpy()
y_train_s5 = train_df[PRIMARY_OUTCOME].to_numpy(dtype=float)

t0 = time.time()
x_tree_model = fit_single_meta_learner('X', X_train_s5, treatment_train_s5, y_train_s5, base_learner_factory=tree_factory)
s_lgbm_model = fit_single_meta_learner('S', X_train_s5, treatment_train_s5, y_train_s5)
propensity_model_s5 = fit_propensity_baseline(train_df, POOLED_TREATMENT_COL, PRIMARY_OUTCOME)
cf_model_s5 = fit_causal_forest(X_train_s5, treatment_train_s5, y_train_s5)
ut_model_s5 = fit_uplift_tree(X_train_s5, treatment_train_s5, y_train_s5)
urf_model_s5 = fit_uplift_random_forest(X_train_s5, treatment_train_s5, y_train_s5)
elapsed = time.time() - t0

scores_val = {
    'X+Tree(depth=4)': predict_single_meta_learner('X', x_tree_model, X_val_s5),
    'S+LightGBM(vanilla)': predict_single_meta_learner('S', s_lgbm_model, X_val_s5),
    'Baseline (propensão)': predict_propensity_score(propensity_model_s5, val_df),
    'CausalForest': predict_causal_forest_uplift(cf_model_s5, X_val_s5),
    'UpliftTree': predict_uplift_tree_uplift(ut_model_s5, X_val_s5),
    'UpliftRF': predict_uplift_random_forest_uplift(urf_model_s5, X_val_s5),
}
fixed_holdout_summary = evaluate_multiple_rankings(val_df[PRIMARY_OUTCOME].values, scores_val, val_df[POOLED_TREATMENT_COL].values)

labels = lang({'header': 'Holdout fixo — train_df completo → val_df'})
print(f"{labels['header']} — tempo total de fit: {elapsed:.1f}s:")
print(fixed_holdout_summary.round(4))


Holdout fixo — train_df completo → val_df — tempo total de fit: 64.5s:
                      qini_auc  uplift_auc  uplift_at_30pct
X+Tree(depth=4)         0.0627      0.0373           0.1014
S+LightGBM(vanilla)     0.0415      0.0244           0.0907
Baseline (propensão)    0.0395      0.0235           0.0923
CausalForest            0.0125      0.0085           0.0592
UpliftTree              0.0105      0.0058           0.0618
UpliftRF                0.0123      0.0075           0.0709


**Insights:**

Os três estimadores especializados de efeito/uplift pontuam bem mais baixo nesta única avaliação de holdout fixo (CausalForest 0,0125; UpliftTree 0,0105; UpliftRF 0,0123) do que suas médias no repeated holdout (0,0131; 0,0254; 0,0201, respectivamente) — o contraste mais marcante é o de UpliftTree, cuja média no repeated holdout foi a maior entre os seis candidatos (0,0254), mas cujo Qini neste único `val_df` é o menor dos seis.

**Esse é o padrão espelhado do que aconteceu com T+Tree em 4.9** (lá: forte no holdout fixo, fraco no repeated holdout; aqui: fraco no holdout fixo, forte no repeated holdout para UpliftTree) — reforça, pelo lado oposto, a mesma lição já registrada no critério de convicção pré-registrado em 5.1: um único resultado de `val_df` não é evidência suficiente, e aqui ele teria sido ativamente enganoso sobre UpliftTree se lido isoladamente, exatamente como antecipado na hipótese registrada antes de rodar. Pela mesma regra pré-registrada ("um resultado espetacular aqui e medíocre no repeated holdout seria lido como instabilidade, não como vitória"), a lógica simétrica se aplica: um resultado fraco isolado em `val_df`, contradito por um sinal comparativamente mais forte no repeated holdout, não deve ser lido como desqualificador. Este número de holdout fixo é reportado como mais um ponto de dado, não como base para ordenar os três estimadores especializados de efeito/uplift entre si.

X+Tree, S+LightGBM e o baseline reproduzem exatamente seus números já conhecidos de S4 (0,0627 / 0,0415 / 0,0395), confirmando que a mecânica do holdout fixo permanece inalterada.

<a id="s5-6"></a>

### 5.6 Correlação entre rankings

Correlação de Spearman entre os scores dos seis candidatos em `val_df`
(mesmos scores de 5.5) — só para ajudar a interpretar por que modelos
diferem (rankings parecidos vs. divergentes), não uma nova seleção.


In [8]:
scores_df = pd.DataFrame(scores_val)
rank_corr = scores_df.corr(method='spearman')

labels = lang({'header': 'Correlação de Spearman entre rankings (val_df)'})
print(f"{labels['header']}:")
print(rank_corr.round(3))


Correlação de Spearman entre rankings (val_df):
                      X+Tree(depth=4)  S+LightGBM(vanilla)  \
X+Tree(depth=4)                 1.000                0.634   
S+LightGBM(vanilla)             0.634                1.000   
Baseline (propensão)            0.258                0.417   
CausalForest                    0.300                0.521   
UpliftTree                      0.601                0.465   
UpliftRF                        0.558                0.548   

                      Baseline (propensão)  CausalForest  UpliftTree  UpliftRF  
X+Tree(depth=4)                      0.258         0.300       0.601     0.558  
S+LightGBM(vanilla)                  0.417         0.521       0.465     0.548  
Baseline (propensão)                 1.000         0.283      -0.043    -0.057  
CausalForest                         0.283         1.000       0.254     0.328  
UpliftTree                          -0.043         0.254       1.000     0.793  
UpliftRF                       

**Insights:**

X+Tree e S+LightGBM — as duas referências de S4 — correlacionam-se mais fortemente entre si (ρ=0,634) do que qualquer outro par não-idêntico, consistente com os dois formarem um top tier em S4. UpliftTree e UpliftRF também se correlacionam fortemente entre si (ρ=0,793) — esperado, já que UpliftRF é um ensemble via bagging de árvores construídas com o mesmo critério de split voltado a uplift.

**UpliftTree e UpliftRF são os dois candidatos com correlação mais fraca — até levemente negativa — com o ranking do baseline de propensão** (ρ=-0,043 e ρ=-0,057, respectivamente): qualitativamente os mais diferentes de um ranking puro de propensão de resposta entre os seis candidatos, consistente com serem estimadores nativamente voltados a uplift, que dividem diretamente por heterogeneidade de efeito em vez de por probabilidade de outcome. Essa distinção qualitativa não implica, por si só, desempenho melhor — dos dois, só UpliftTree mostrou uma vantagem pareada consistente sobre o baseline em 5.4 — mas ajuda a explicar por que o perfil de ranking de UpliftTree diverge do baseline mais do que os das referências de S4.

CausalForest correlaciona-se mais com S+LightGBM (ρ=0,521) do que com qualquer outro candidato — ambos passam por alguma forma de regressão de outcome/efeito, o que plausivelmente explica parte dessa estrutura de ranking compartilhada, apesar do desempenho absoluto mais fraco de CausalForest.

<a id="s5-7"></a>

### 5.7 Síntese da S5

**Nenhum dos três estimadores especializados de efeito/uplift estabeleceu uma vantagem estável sobre as referências congeladas de S4.** X-learner + árvore rasa e S-learner + LightGBM vanilla continuam no top tier ao final desta rodada — nenhuma evidência recolhida em S5 os elimina, e nenhuma delas os supera de forma consistente.

**UpliftTree é o caso mais interessante desta seção, mas não um vencedor confirmado.** Teve a maior média (0,0254) e a maior taxa de vitória (40,0%) entre os seis candidatos no repeated holdout. UpliftTree apresentou a maior proporção observada de deltas positivos contra o baseline neste protocolo, 10 de 15 reamostragens (66,67%, delta mediano +0,0077) — apenas uma repetição acima dos 9 de 15 (60%) observados para S+LightGBM contra esse mesmo baseline em S4 — mas ficou sem separação estável, no protocolo repetido, de X+Tree (delta médio +0,0004, 8 de 15 splits, 53,33%) e teve o pior resultado de todos os seis no único holdout fixo (0,0105), um padrão de sensibilidade a protocolo que espelha, na direção oposta, o já visto com T+Tree em 4.9. Sua mediana (0,0201), mais baixa que sua média, sugere uma distribuição assimétrica — vale carregar essa ressalva adiante, não só o número de maior destaque.

**UpliftRF apresenta o perfil mais estável numericamente (menor desvio-padrão, 0,0096) mas nunca foi o melhor candidato em nenhum dos 15 splits.** Fica perto do baseline em média (0,0201 vs. 0,0209) e o supera em 60,0% dos splits, um sinal moderado, mais fraco que o de UpliftTree.

**CausalForest foi o candidato mais fraco desta rodada em ambos os protocolos** — menor média no repeated holdout (0,0131), sem vantagem pareada positiva contra nenhuma das três referências, e o segundo pior resultado no holdout fixo. Isso não invalida a família — reflete o desempenho desta configuração pré-registrada e conservadora (`model_t` X-invariante, hiperparâmetros default) neste dataset, não uma avaliação geral de Causal Forests.

**Resposta à pergunta central de S5 — os estimadores especializados de efeito/uplift superam de forma consistente as referências de S4 ou o baseline de propensão?** Não, com os dados e a configuração avaliados até aqui — com a ressalva de que UpliftTree mostra um sinal direcionalmente positivo e não-trivial especificamente contra o baseline de propensão, que vale carregar como candidato para revisão, não como vencedor confirmado.

**Limitações.** O custo computacional do `CausalForestDML` — ~207s por fit em smoke test isolado — motivou cautela quanto ao orçamento desta rodada; na execução completa final de 5.3, os 15 repeated holdouts com os seis candidatos totalizaram ao todo ~781,5s (~13 min). Esse custo limitou esta rodada à configuração default pré-registrada — não foi testada nenhuma configuração alternativa, tunada ou não, dentro do orçamento desta rodada (Regra Absoluta #6). A assimetria da distribuição de UpliftTree (mediana abaixo da média) não foi investigada mais a fundo. Como em S4, os 15 splits do repeated holdout se sobrepõem parcialmente — não são amostras independentes; nenhum teste de significância formal foi calculado aqui, e não seria a base apropriada de decisão mesmo se calculado.

**Estado ao final de S5:** X+Tree(depth=4), S+LightGBM(vanilla) e o baseline de propensão seguem como referências; UpliftTree entra como candidato adicional a revisar, sem substituir nenhum dos anteriores; UpliftRF e CausalForest não mostraram evidência competitiva nesta rodada. **Nenhuma configuração final foi escolhida.** A seleção entre esses candidatos fica para revisão explícita antes de congelar a configuração e abrir o teste selado em S6, seguindo o protocolo definido ao final de S4.

**Próximo:** revisão dos resultados de S5 antes de prosseguir — S6 não foi iniciada.

<a id="s5-8"></a>

### 5.8 Decisão pré-S6 — tomada após revisão de S5 e antes da abertura do teste

Esta decisão foi tomada usando exclusivamente dados de desenvolvimento (`train_df`/`val_df`). **O teste selado ainda não foi aberto.**

**Modelo primário da S6:** `UpliftTreeClassifier`, exatamente com a configuração usada em S5 (`control_name='control'`, `random_state=SEED`, demais hiperparâmetros no default da biblioteca).

**Comparadores pré-especificados:**
1. X-learner + `DecisionTreeRegressor(max_depth=4)`
2. S-learner + LightGBM vanilla
3. Baseline de propensão de resposta

**Não levados para a avaliação principal de S6:** UpliftRandomForest, CausalForestDML. A exclusão deles não significa que essas famílias sejam inadequadas em geral — significa apenas que as configurações pré-registradas avaliadas em S5 não apresentaram evidência competitiva suficiente para justificar aumentar a multiplicidade da avaliação selada.

**Justificativa para UpliftTree** (números do repeated holdout de S5, 15 repetições, dentro de `train_df`):

- maior Qini médio entre os seis candidatos: 0,0254;
- maior win rate global entre os seis: 40,0%;
- 10 de 15 deltas positivos contra o baseline de propensão;
- delta mediano contra o baseline: +0,0077;
- empate prático/descritivo com X+Tree: delta médio +0,0004, delta mediano +0,0001, 8 de 15 vitórias (53,33%);
- ausência de separação estável contra S+LightGBM (5.4);
- resultado fraco no holdout fixo (Qini 0,0105) — mantido explicitamente como alerta de sensibilidade a amostra/protocolo, não descartado nem escondido.

**Esta escolha não deve ser descrita como evidência de superioridade estatística.** É uma seleção de modelo primário baseada no protocolo de desenvolvimento já privilegiado ao longo do projeto (repeated holdout), acrescida de um critério de desempate favorável à simplicidade e interpretabilidade de uma árvore especializada em uplift.

Esta decisão fica registrada antes da abertura do teste selado — ver `artifacts/s6/preregistration.json`.
